# Lab 3 — Reflexion: learning from mistakes without training (§19.2.3)

A good agent must be able to stop and evaluate itself. The textbook splits reflection into 3 levels:
1. **Output validation** — "Is it correct? Did I forget anything?"
2. **Trajectory review** — look back over the last *k* steps
3. **Strategy revision** — "Am I solving the right problem?"

**Reflexion** keeps lessons from mistakes in a *reflection memory* and puts them into the next attempt's prompt
→ it learns across attempts **without updating any weights**

> Reference: Haggai Roitman, *The Hitchhiker's Guide to Agentic AI*, §19.2.3 — CC BY-SA 4.0

> ✏️ **Starter notebook — you write the key parts**
>
> Most of the code is ready. Write only the **5 places marked `✏️ TODO`** (replace `___` or `...` with your code). Watch the video first, then try it yourself without opening the solution.
>
> Run every cell from top to bottom. A `NameError` or `TypeError` in a cell you have not finished yet is normal.
>
> Stuck? The solution is **`ep05-lab3-reflexion.en.ipynb`** (the same code as in the video).

## 0. Setup

You need Python 3.10+ (or Colab) and an **OpenRouter API key** — setup instructions are in Lab 1

In [ ]:
%pip install -q "openai>=1.50" python-dotenv

In [ ]:
# ---- from Lab 1: set up client + ask_llm ----
import os

from openai import OpenAI

def load_key():
    try:
        from google.colab import userdata
        return userdata.get("OPENROUTER_API_KEY")
    except ImportError:
        from dotenv import load_dotenv
        load_dotenv()
        return os.environ["OPENROUTER_API_KEY"]

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=load_key())
MODEL = "google/gemma-4-26b-a4b-it"
EXTRA = {"provider": {"require_parameters": True}, "reasoning": {"effort": "medium"}}

def ask_llm(prompt):
    resp = client.chat.completions.create(model=MODEL, extra_body=EXTRA,
                                          messages=[{"role": "user", "content": prompt}])
    return resp.choices[0].message.content

## 1. Task + Evaluator

The agent must write a `parse_duration()` function, and an **evaluator** checks it with unit tests the agent **cannot see in advance**
(like real work, where some requirements aren't written in the task)

In [ ]:
SPEC = ("Write a Python function `parse_duration(s: str) -> float` that converts a duration "
        "string like '1h30m' into minutes (e.g. '1h30m' -> 90). Return only the code in one "
        "```python block, no explanation.")

In [ ]:
HIDDEN_TESTS = [
    ("1h30m", 90), ("2h", 120), ("45m", 45),
    ("90s", 1.5),         # seconds
    ("1H 15M", 75),       # uppercase + spaces
    ("1.5h", 90),         # decimals
    ("1d", 1440),         # days
    ("abc", ValueError),  # invalid -> must raise ValueError
    ("10", ValueError),   # no unit -> ambiguous -> ValueError
]

In [ ]:
def extract_code(text):
    return text.split("```python")[-1].split("```")[0] if "```" in text else text

In [ ]:
# ✏️ TODO 1: Run the model's code in the namespace ns, then pull out the parse_duration function
def evaluate(code_str):
    ns = {}
    try:
        ___
        fn = ___
    except Exception as e:
        return [f"code does not run: {e}"]
    failures = []
    for inp, want in HIDDEN_TESTS:
        expects_error = isinstance(want, type) and issubclass(want, Exception)
        try:
            got = fn(inp)
            if expects_error:
                failures.append(f"parse_duration({inp!r}) returned {got}, expected {want.__name__}")
            elif abs(got - want) > 1e-9:
                failures.append(f"parse_duration({inp!r}) returned {got}, expected {want}")
        except Exception as e:
            if not (expects_error and isinstance(e, want)):
                failures.append(f"parse_duration({inp!r}) raised {type(e).__name__}: {e}")
    return failures

## 2. Reflexion loop

`attempt → evaluate → reflect → (save to memory) → new attempt that reads the memory`

In [ ]:
# ✏️ TODO 2: Append every lesson to the prompt, one per line, each starting with "- "
def write_code(reflections):
    prompt = SPEC
    if reflections:
        prompt += "\n\nLessons from your previous failed attempts:\n"
        prompt += ___
    return extract_code(ask_llm(prompt))

In [ ]:
# ✏️ TODO 3: Send the code + failing tests to the model and ask for a one-sentence lesson starting with 'I failed because'
def reflect(code_str, failures):
    return ask_llm(___).strip()

In [ ]:
# ✏️ TODO 4: Each round: write code using the memory, then evaluate it with the hidden tests
def reflexion(max_attempts=4):
    memory = []
    for attempt in range(1, max_attempts + 1):
        code_str = ___
        failures = ___
        print(f"\n🧪 attempt {attempt}: {len(HIDDEN_TESTS) - len(failures)}/{len(HIDDEN_TESTS)} tests passed")
        if not failures:
            print("✅ all tests passed")
            return code_str, memory
        # ✏️ TODO 5: Reflect on the failures to get a lesson, then store it in memory for the next round
        for f in failures:
            print("   ❌", f)
        lesson = ___
        ___
        print("   📝", lesson)
    return code_str, memory

In [ ]:
final_code, memory = reflexion()
print("\n📓 REFLECTION MEMORY:")
for m in memory:
    print(" -", m)

In [ ]:
print(final_code)

## Summary

| Component | In the code |
|---|---|
| Evaluator | `evaluate()` — hidden unit tests |
| Reflection | `reflect()` — a one-sentence lesson "I failed because…" |
| Reflection memory | `memory` — accumulates every round, put into the prompt via `write_code()` |
| Guard | `max_attempts` |

**Try next:** call `write_code([])` every round (no memory) and compare the number of attempts needed — the effect of reflection memory becomes clear